## Installments Payments Feature Engineering

In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [2]:
from src.data_utils import load_raw, save_interim
import numpy as np
import pandas as pd

In [3]:
df_installments=load_raw('installments_payments.csv')

### Delay Features

In [4]:
inst = df_installments.copy()
inst['PAYMENT_DELAY'] = inst['DAYS_ENTRY_PAYMENT']-inst['DAYS_INSTALMENT']
inst['LATE'] = (inst['PAYMENT_DELAY'] < 0).astype(int)
inst['SEVERE_LATE'] = (inst['PAYMENT_DELAY'] < -30).astype(int)

### Underpayment

In [5]:
inst['UNDERPAYMENT'] = (inst['AMT_PAYMENT'] < inst['AMT_INSTALMENT']).astype(int)

instalment_agg = inst.groupby('SK_ID_CURR').agg(
    INST_TOTAL_RECORDS = ('SK_ID_PREV', 'count'),
    INST_UNIQUE_LOANS = ('SK_ID_PREV', 'nunique'),
    INST_AVG_DELAY = ('PAYMENT_DELAY', 'mean'),
    INST_MAX_DELAY = ('PAYMENT_DELAY', 'max'),
    INST_LATE_COUNT = ('LATE', 'sum'),
    INST_SEVERE_LATE_COUNT = ('SEVERE_LATE', 'sum'),
    INST_UNDERPAYMENT_COUNT = ('UNDERPAYMENT', 'sum'),
    INST_AVG_INSTALMENT = ('AMT_INSTALMENT', 'mean'),
    INST_AVG_PAYMENT = ('AMT_PAYMENT', 'mean')
)

### Ratio

In [7]:
instalment_agg['INST_LATE_RATIO'] = (
    instalment_agg['INST_LATE_COUNT'] / instalment_agg['INST_TOTAL_RECORDS']
)

instalment_agg['INST_SEVERE_LATE_RATIO'] = (
    instalment_agg['INST_SEVERE_LATE_COUNT'] / instalment_agg['INST_TOTAL_RECORDS']
)

instalment_agg['INST_UNDERPAYMENT_RATIO'] = (
    instalment_agg['INST_UNDERPAYMENT_COUNT'] / instalment_agg['INST_TOTAL_RECORDS']
)

In [8]:
save_interim(instalment_agg, 'installments_agg.csv')